# Linear Combination

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from dataclasses import asdict

import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.linear_combination_catalog import (
    linear_combination_example,
    linear_combination_example_circuit,
    linear_combination_example_model,
)
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_comparison import count_emitted_instructions
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.runtime.dynamic.execution import run_dynamic

## Linear combination of block encodings

Given child blocks for $U_i/\alpha_i$, the signed combination for
$\sum_i\lambda_i U_i$ has normalization $\sum_i|\lambda_i|\alpha_i$.
The ancilla width is $\lceil\log_2 n\rceil+\max_i a_i$.
The additional resource register holds child resources and the coefficient
PREPARE junk.

PREPARE samples the normalized magnitudes using the specified coefficient
precision. SELECT applies the indexed child with the coefficient sign and
PREPARE adjoint completes the zero-signal block. Custom preparation and
selection can be supplied at the existing oracle boundaries. A positive
outer control changes the SELECT stage and leaves the two preparation
stages in place. The native metadata evaluates the error as the maximum child error times the sum of the
normalized coefficient magnitudes, as in the pinned implementation.

[Quantum algorithms: A survey of applications and end-to-end complexities](https://arxiv.org/abs/2310.03011)
describes the LCU construction.

In [ ]:
from flagquantum.algorithms.primitives.linear_combination_block_encoding import (
    LinearCombinationBlockEncoding,
)

### Example Instances

In [ ]:
encoding = linear_combination_example()
assert isinstance(encoding, LinearCombinationBlockEncoding)
assert encoding.coefficients == (0.25, -0.25, 0.25, -0.25)
assert encoding.alpha == 1

#### Graphical Signature

In [ ]:
signature = Signature.build(
    system=encoding.system_bitsize,
    ancilla=encoding.ancilla_bitsize,
    resource=encoding.resource_bitsize,
)
display(SVG(field_signature_svg([("Linear combination", signature, {})])))

### Call Graph

In [ ]:
model = linear_combination_example_model()
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name,
)))

### Verify the appended block

For the original T, Hadamard, X and Z example, project the output onto the
zero ancilla and resource sector. Compare its complete complex system
amplitudes and verify the private work bits are zero.

In [ ]:
circuit = linear_combination_example_circuit()
ir = circuit.to_ir()
shots = run_dynamic(circuit, shots=2, seed=11, strategy="trajectory")
states = shots.final_states.reshape(2, -1).to(torch.complex128)
width = circuit.n_qubits
stride = 1 << (width - 1)
h = torch.tensor([1, 1], dtype=torch.complex128) / 2**0.5
expected = (torch.tensor([0, 1], dtype=torch.complex128) - h) / 4
torch.testing.assert_close(states[:, [0, stride]], expected.expand(2, 2), atol=4e-12, rtol=0)
private_width = width - encoding.system_bitsize - encoding.ancilla_bitsize - encoding.resource_bitsize
work = states.reshape(2, -1, 1 << private_width)
torch.testing.assert_close(work[:, :, 1:], torch.zeros_like(work[:, :, 1:]), atol=4e-12, rtol=0)
physical = count_emitted_instructions(ir)
logical = estimate_component_resources(model)
print("Emitted T/T-dagger instructions", physical.t_count)
print("Emitted measurements", physical.unconditional["measure"])
print("Source resource model", asdict(logical.gates))